# E2 · Ablation Table 4 trên CIFAR-100
Import notebook và gắn dataset backbone ba seed của nhóm. Bật Internet, chọn GPU và cấp quyền cho secret `WANDB_API_KEY`. Gắn thêm `fecam-e2-notebook.zip` làm Kaggle Input để dùng bản source đi kèm. Nếu không gắn source, notebook clone nhánh `feat/e2-table4-ablation` trên GitHub. Các cell bên dưới định nghĩa 5 biến thể, chạy test rồi mới bắt đầu đánh giá.


In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys
import tempfile

BRANCH = "feat/e2-table4-ablation"
REMOTE = "https://github.com/duongthienphu/fecam-study.git"
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
WORK.mkdir(parents=True, exist_ok=True)
from zipfile import ZipFile
import hashlib

archives = list(INPUT.rglob("fecam-e2-notebook.zip"))
roots = [p.parents[2] for p in INPUT.rglob("src/models/fecam.py")
         if (p.parents[2] / "notebooks/kaggle_e2_table4_ablation.ipynb").is_file()]
if len(archives) + len(roots) > 1:
    raise ValueError("Chỉ gắn một bản source E2")
if roots:
    REPO = roots[0]
elif archives:
    REPO = Path(tempfile.mkdtemp(prefix="e2-code-", dir=WORK))
    with ZipFile(archives[0]) as archive:
        archive.extractall(REPO)
else:
    REPO = Path(tempfile.mkdtemp(prefix="e2-code-", dir=WORK))
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REMOTE, str(REPO)], check=True)
if archives or roots:
    commit = None
else:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
source_sha = hashlib.sha256((REPO / "src/models/fecam.py").read_bytes()).hexdigest()
print("Source:", REPO, "FeCAM SHA256:", source_sha)
os.chdir(WORK)
sys.path.insert(0, str(REPO))
missing = [pkg for mod, pkg in [("datasets", "datasets"), ("wandb", "wandb"), ("yaml", "PyYAML")]
           if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)


In [ ]:
"""Classifier settings and reference percentages for the CIFAR-100 Table 4 experiment."""

from dataclasses import dataclass
from statistics import mean

from src.models.fecam import FeCAM


@dataclass(frozen=True)
class Variant:
    covariance: str
    transform: str
    correlation: bool
    paper_avg: float
    paper_final: float

    def classifier(self, device):
        return FeCAM(covariance_mode=self.covariance, transform=self.transform,
                     correlation_normalization=self.correlation, device=device)


VARIANTS = {
    "ncm": Variant("none", "none", False, 64.8, 51.6),
    "ncm-tukey": Variant("none", "sqrt", False, 66.6, 54.4),
    "fecam-full": Variant("classwise", "sqrt", True, 70.9, 62.1),
    "fecam-no-tukey": Variant("classwise", "none", True, 65.9, 55.4),
    "fecam-no-norm": Variant("classwise", "sqrt", False, 62.8, 52.1),
}


def summarize(name, seed, sessions):
    if [row["num_seen_classes"] for row in sessions] != [50, 60, 70, 80, 90, 100]:
        raise ValueError("Expected all six B50-I10 sessions in order")
    scores = [row["seen_acc_pct"] for row in sessions]
    if not all(0 <= score <= 100 for score in scores):
        raise ValueError("Expected percentage accuracies")
    setting = VARIANTS[name]
    avg, final = mean(scores), scores[-1]
    return dict(run=f"{name}__seed{seed}", seed=seed, variant=name,
                avg_acc_pct=avg, final_acc_pct=final,
                paper_avg_acc_pct=setting.paper_avg, paper_final_acc_pct=setting.paper_final,
                avg_delta_pp=avg - setting.paper_avg, final_delta_pp=final - setting.paper_final)


In [ ]:
import unittest

import torch




class TestTable4(unittest.TestCase):
    def test_component_settings(self):
        settings = [(v.covariance, v.transform, v.correlation) for v in VARIANTS.values()]
        self.assertEqual(settings, [
            ("none", "none", False), ("none", "sqrt", False), ("classwise", "sqrt", True),
            ("classwise", "none", True), ("classwise", "sqrt", False),
        ])

    def test_incremental_predictions(self):
        base = torch.tensor([[4., 0.1], [3.8, 0.2], [4.1, 0.3]])
        for name, variant in VARIANTS.items():
            with self.subTest(name=name):
                model = variant.classifier("cpu")
                model.update(base, torch.zeros(3, dtype=torch.long))
                model.update(base.flip([1]), torch.ones(3, dtype=torch.long))
                self.assertEqual(model.predict(torch.stack([base[0], base[0].flip([0])])).tolist(),
                                 [0, 1])

    def test_metrics(self):
        sessions = [dict(num_seen_classes=n, seen_acc_pct=130-n) for n in range(50, 101, 10)]
        row = summarize("ncm", 2, sessions)
        self.assertEqual((row["avg_acc_pct"], row["final_acc_pct"]), (55, 30))
        self.assertAlmostEqual(row["avg_delta_pp"], -9.8)
        with self.assertRaises(ValueError):
            summarize("ncm", 2, sessions[:-1])

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTable4)
if not unittest.TextTestRunner(verbosity=2).run(suite).wasSuccessful():
    raise RuntimeError("E2 tests failed")


In [ ]:
import json
import pandas as pd
import torch
import wandb
from kaggle_secrets import UserSecretsClient
from scripts.run_experiment import (checkpoint_sha256, extract_features, load_checkpoint,
                                    load_cifar100, load_config, make_loader, seed_everything,
                                    TaskSplit, validate_checkpoint)

PROJECT = "fecam-study"
ENTITY = "duongthienphu-university-of-science-vnuhcm"
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()
device = "cuda" if torch.cuda.is_available() else "cpu"
seeds = (1, 2, 3)
paths = {}
for seed in seeds:
    matches = sorted(INPUT.rglob(f"backbone-seed{seed}/base.pt"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one backbone-seed{seed}/base.pt, found {matches}")
    paths[seed] = matches[0]
out = Path(tempfile.mkdtemp(prefix="e2-results-", dir=WORK))
data = load_cifar100(WORK / "cifar-data")
print(device, out)

In [ ]:
def evaluate_seed(seed):
    # E1 and E2 use the same frozen CIFAR-100 B50-I10 backbone protocol.
    cfg = load_config(REPO / "configs/experiments/e1_fecam_classwise.yaml")
    cfg["seed"] = seed
    seed_everything(seed)
    split = TaskSplit(**cfg["split"])
    backbone, metadata = load_checkpoint(paths[seed], device)
    validate_checkpoint(metadata, cfg, split)
    models = {name: spec.classifier(device) for name, spec in VARIANTS.items()}
    histories = {name: [] for name in models}
    for session in range(len(split.tasks)):
        train_loader = make_loader(data, split, cfg, session, "train", device)
        test_loader = make_loader(data, split, cfg, session, "test", device, seen=True)
        train_x, train_y = extract_features(backbone, train_loader, device)
        test_x, test_y = extract_features(backbone, test_loader, device)
        for name, model in models.items():
            model.update(train_x, train_y)
            preds = torch.cat([model.predict(batch).cpu() for batch in test_x.split(128)])
            correct = (preds == test_y).sum().item()
            histories[name].append(dict(session=session, num_seen_classes=model.num_classes,
                                        seen_acc_pct=100 * correct / len(test_y)))
        print(f"seed={seed} session={session} classes={models['ncm'].num_classes}", flush=True)
    return histories, {"git_commit": commit, "fecam_sha256": source_sha, "seed": seed, "protocol": cfg["protocol"],
                       "class_order": split.class_order, "checkpoint_sha256": checkpoint_sha256(paths[seed])}


rows = []
for seed in seeds:
    histories, provenance = evaluate_seed(seed)
    for name, sessions in histories.items():
        row = summarize(name, seed, sessions)
        spec = VARIANTS[name]
        with wandb.init(project=PROJECT, entity=ENTITY, name=row["run"], group=out.name,
                        tags=["E2", "table4-ablation"], dir=str(out),
                        config={**provenance, "variant": name, "transform": spec.transform,
                                "covariance_mode": spec.covariance,
                                "correlation_normalization": spec.correlation}) as run:
            for record in sessions:
                run.log(record, step=record["session"])
            run.summary.update({k: v for k, v in row.items() if k not in ("run", "seed", "variant")})
            row["wandb_url"] = run.url
        rows.append(row)
    (out / f"seed{seed}-sessions.json").write_text(json.dumps(
        {"provenance": provenance, "sessions": histories}, indent=2))
    pd.DataFrame(rows).to_csv(out / "table4_runs.csv", index=False)

In [ ]:
runs = pd.DataFrame(rows)
if len(runs) != len(seeds) * len(VARIANTS) or runs.duplicated(["variant", "seed"]).any():
    raise ValueError("Incomplete or duplicated Table 4 runs")
table = runs.groupby("variant", sort=False).agg(
    avg_mean=("avg_acc_pct", "mean"), avg_std=("avg_acc_pct", "std"),
    final_mean=("final_acc_pct", "mean"), final_std=("final_acc_pct", "std"),
    paper_avg=("paper_avg_acc_pct", "first"), paper_final=("paper_final_acc_pct", "first"),
    avg_delta_pp=("avg_delta_pp", "mean"), final_delta_pp=("final_delta_pp", "mean")
).reset_index()
table.to_csv(out / "table4_comparison.csv", index=False)
with wandb.init(project=PROJECT, entity=ENTITY, name="E2-table4-comparison", group=out.name,
                tags=["E2", "table4-ablation"], dir=str(out), config={"git_commit": commit, "fecam_sha256": source_sha}) as run:
    run.log({"table4": wandb.Table(dataframe=table)})
    artifact = wandb.Artifact("e2-table4-results", type="evaluation")
    for path in sorted(out.glob("*")):
        if path.suffix in {".csv", ".json"}:
            artifact.add_file(str(path))
    run.log_artifact(artifact)
print(out)
display(table.round(3))